In [4]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
import vectorbt as vbt
import sys

sys.path.insert(0, str(Path('.').resolve().parent))
from dema_strat import load_ohlcv, vbt_frame, free

CASH = 50_000
COMMISSION = 0.00015
TABLE_NAME = "f0"

def load_data(file_path, *, verbose=False, use_vwap=False):
    # Compact float32 + Parquet-cached loader.
    return load_ohlcv(file_path, use_cache=True, add_vwap=use_vwap, verbose=verbose)

def make_signals(data, n1=9, n2=21, atr_period=14, sl_mult=1.0, rr_ratio=0.5, use_vwap=False):
    # keep float32 through indicator math; cast up only at the vectorbt boundary
    close = data["Close"]
    high = data["High"]
    low = data["Low"]

    ema1 = vbt.indicators.MA.run(close, window=n1, ewm=True)
    ema2 = vbt.indicators.MA.run(close, window=n2, ewm=True)
    atr = vbt.indicators.ATR.run(high, low, close, window=atr_period)

    sl_dist = sl_mult * atr.atr
    tp_dist = rr_ratio * sl_dist
    sl_pct = sl_dist / close
    tp_pct = tp_dist / close

    long_mask = (ema1.ma > ema2.ma) & np.isfinite(atr.atr) & (atr.atr > 0)
    short_mask = (ema2.ma > ema1.ma) & np.isfinite(atr.atr) & (atr.atr > 0)

    if use_vwap:
        long_mask = long_mask & (close > data["VWAP"])
        short_mask = short_mask & (close < data["VWAP"])
    long_signal = long_mask
    short_signal = short_mask

    return pd.DataFrame({
        "ema1": ema1.ma,
        "ema2": ema2.ma,
        "atr": atr.atr,
        "sl_pct": sl_pct,
        "tp_pct": tp_pct,
        "long": long_signal,
        "short": short_signal,
    })

def run_backtest(data, n1=9, n2=21, atr_period=14, sl_mult=1.0, rr_ratio=0.5, use_vwap=False):
    signals = make_signals(data, n1=n1, n2=n2, atr_period=atr_period, sl_mult=sl_mult, rr_ratio=rr_ratio, use_vwap=use_vwap)
    freq = data.index.to_series().diff().median()

    long_signal  = signals["long"].to_numpy()
    short_signal = signals["short"].to_numpy()
    # vbt wants float64 for size/close; cast just the columns it needs.
    close_np = data["Close"].to_numpy(dtype=np.float64)

    portfolio = vbt.Portfolio.from_signals(
        close=close_np,
        entries=long_signal,
        exits=short_signal,
        short_entries=short_signal,
        short_exits=long_signal,
        init_cash=CASH,
        fees=COMMISSION,
        size=100.0,
        size_type="value",
        freq=freq,
        sl_stop=signals["sl_pct"].to_numpy(dtype=np.float64),
        tp_stop=signals["tp_pct"].to_numpy(dtype=np.float64),
    )

    return portfolio, portfolio.stats(), signals

def run_params(data, n1, n2, atr_period, sl_mult, rr_ratio, use_vwap=False, asset=None, timeframe=None):
    try:
        portfolio, stats, _ = run_backtest(data, n1=n1, n2=n2, atr_period=atr_period, sl_mult=sl_mult, rr_ratio=rr_ratio, use_vwap=use_vwap)
        return {
            "n1": n1, "n2": n2, "atr_period": atr_period,
            "sl_mult": sl_mult, "rr_ratio": rr_ratio,
            "use_vwap": use_vwap,
            "commission": COMMISSION,
            "asset": asset,
            "timeframe": timeframe,
            "return_pct": float(stats["Total Return [%]"]) if stats.get("Total Return [%]") is not None else None,
            "max_dd": float(stats["Max Drawdown [%]"]) if stats.get("Max Drawdown [%]") is not None else None,
            "sharpe": float(stats["Sharpe Ratio"]) if stats.get("Sharpe Ratio") is not None else None,
            "win_rate": float(stats["Win Rate [%]"]) if stats.get("Win Rate [%]") is not None else None,
            "profit_factor": float(stats["Profit Factor"]) if stats.get("Profit Factor") is not None else None,
            "expectancy": float(stats["Expectancy"]) if stats.get("Expectancy") is not None else None,
            "trades": int(stats["Total Trades"]) if stats.get("Total Trades") is not None else None,
        }
    except Exception as e:
        return {
            "n1": n1, "n2": n2, "atr_period": atr_period,
            "sl_mult": sl_mult, "rr_ratio": rr_ratio,
            "use_vwap": use_vwap,
            "commission": COMMISSION,
            "asset": asset,
            "timeframe": timeframe,
            "return_pct": None, "max_dd": None, "sharpe": None,
            "win_rate": None, "profit_factor": None,
            "expectancy": None, "trades": None,
        }


In [2]:
file_path = Path("~/market_data/XAUUSD/XAUUSD_M1.csv").expanduser()
data = load_data(file_path, verbose=True)

portfolio, stats, signals = run_backtest(data, rr_ratio=1)
print(stats)

trades = portfolio.trades.records_readable
print(trades)

free(data, signals)

[loader] cache hit: XAUUSD_M1__cache_v1.parquet
Start                                                 0
End                                             2088752
Period                               1450 days 12:33:00
Start Value                                     50000.0
End Value                                  37946.869021
Total Return [%]                             -24.106262
Benchmark Return [%]                         184.185378
Max Gross Exposure [%]                         0.263868
Total Fees Paid                             11839.71693
Max Drawdown [%]                              24.106339
Max Drawdown Duration                1450 days 12:05:00
Total Trades                                     394657
Total Closed Trades                              394657
Total Open Trades                                     0
Open Trade PnL                                      0.0
Win Rate [%]                                    28.9051
Best Trade [%]                                 5.501654


In [ ]:
portfolio.plot()

## Optimization

In [ ]:

from itertools import product
import time


db_path = Path("../backtest_results.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")
cursor.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    n1 INTEGER NOT NULL,
    n2 INTEGER NOT NULL,
    atr_period INTEGER NOT NULL,
    sl_mult REAL NOT NULL,
    rr_ratio REAL NOT NULL,
    use_vwap INTEGER NOT NULL,
    commission REAL NOT NULL,
    asset TEXT NOT NULL,
    timeframe TEXT NOT NULL,
    expectancy REAL,
    win_rate REAL,
    max_dd REAL,
    sharpe_ratio REAL,
    profit_factor REAL,
    num_trades INTEGER,
    return_pct REAL,
    run_timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
""")
conn.commit()

grid = list(product(
    ["M1", "M5", "M15"], # timeframes
    range(5, 15, 2), # fast ema
    range(18, 30, 2), # slow ema
    [14, 18, 21], # atr period
    [1.0, 2.0, 3.0], # sl mult
    [0.25, 0.5, 1.0, 1.5, 2.0], # rr ratio
    [False, True], # use vwap
))

#* Only for test
# grid = list(product(
#     ["M1"], # timeframes
#     [9], # fast ema
#     [21], # slow ema
#     [14], # atr period
#     [1], # sl mult
#     [1], # rr ratio
#     [True, False], # use vwap
# ))

t0 = time.time()

print(f"[{time.strftime('%H:%M:%S')}] Testing {len(grid)} combinations -> {TABLE_NAME}")
asset = "XAUUSD"
# Load each timeframe once and reuse across all param combos (big memory win on M1).
data_by_tf: dict[tuple[str, bool], pd.DataFrame] = {}
for tf in sorted({g[0] for g in grid}):
    for uv in sorted({g[6] for g in grid if g[0] == tf}):
        fp = Path(f"~/market_data/{asset}/{asset}_{tf}.csv").expanduser()
        data_by_tf[(tf, uv)] = load_data(fp, verbose=True, use_vwap=uv)

ONE_LOOP_MEAN_TIME = 0.0
for i, (timeframe, n1, n2, ap, sm, rr, uv) in enumerate(grid):

    data = data_by_tf[(timeframe, uv)]

    t1 = time.time()
    result = run_params(data, n1, n2, ap, sm, rr, use_vwap=uv, asset=asset, timeframe=timeframe)
    elapsed = time.time() - t1
    cursor.execute(
        f"INSERT INTO {TABLE_NAME} (n1, n2, atr_period, sl_mult, rr_ratio, use_vwap, commission, asset, timeframe, expectancy, win_rate, max_dd, sharpe_ratio, profit_factor, num_trades, return_pct) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        (result["n1"], result["n2"], result["atr_period"], result["sl_mult"], result["rr_ratio"], uv, result["commission"], result["asset"], result["timeframe"],
         result["expectancy"], result["win_rate"], result["max_dd"], result["sharpe"], result["profit_factor"], result["trades"], result["return_pct"]),
    )
    conn.commit()
    ONE_LOOP_MEAN_TIME = (elapsed + ONE_LOOP_MEAN_TIME * i) / (i + 1) if i else elapsed
    combinations_left = len(grid) - (i + 1)
    est_left_sec = ONE_LOOP_MEAN_TIME * combinations_left
    if (i + 1) % max(1, len(grid) // 100) == 0 or (i + 1) == len(grid):
        print(
            f"[{time.strftime('%H:%M:%S')}] {i+1:>5}/{len(grid)} tf={timeframe} n1={n1} n2={n2} ap={ap} sm={sm} rr={rr} vwap={uv} "
            f"| sharpe={result['sharpe']} ret={result['return_pct']} trades={result['trades']} ({elapsed:.1f}s) "
            f"| mean={ONE_LOOP_MEAN_TIME:.1f}s est_left={est_left_sec/60:.1f}min"
        )

    free()

free(data_by_tf)

sweep_df = pd.read_sql_query(f"SELECT * FROM {TABLE_NAME}", conn)
sweep_df = sweep_df.dropna(subset=["sharpe_ratio"])
sweep_df = sweep_df.sort_values("sharpe_ratio", ascending=False).reset_index(drop=True)
print(f"Total grid: {len(grid)} | Non-null: {len(sweep_df)}")
sweep_df.head(20)



[13:40:15] Testing 8100 combinations -> f0
[loader] cache hit: XAUUSD_M1__cache_v1.parquet
[loader] cache hit: XAUUSD_M1__cache_v1.parquet
[loader] cache hit: XAUUSD_M15__cache_v1.parquet
[loader] cache hit: XAUUSD_M15__cache_v1.parquet
[loader] cache hit: XAUUSD_M5__cache_v1.parquet
[loader] cache hit: XAUUSD_M5__cache_v1.parquet
[13:41:44]    81/8100 tf=M1 n1=5 n2=18 ap=21 sm=3.0 rr=0.25 vwap=False | sharpe=-134.36934835195223 ret=-21.48947886380047 trades=348369 (1.0s) | mean=0.9s est_left=115.6min
[13:43:15]   162/8100 tf=M1 n1=5 n2=20 ap=21 sm=2.0 rr=0.25 vwap=True | sharpe=-116.9518305911966 ret=-15.525071340453941 trades=255122 (0.9s) | mean=0.9s est_left=115.2min
[13:44:45]   243/8100 tf=M1 n1=5 n2=22 ap=21 sm=1.0 rr=0.5 vwap=False | sharpe=-223.29983903051098 ret=-32.85936363744057 trades=538359 (1.1s) | mean=0.9s est_left=114.0min
[13:46:16]   324/8100 tf=M1 n1=5 n2=24 ap=18 sm=3.0 rr=0.5 vwap=True | sharpe=-49.42882136859572 ret=-7.342282069191502 trades=118248 (0.8s) | mean

,id,n1,n2,atr_period,sl_mult,rr_ratio,use_vwap,commission,asset,timeframe,expectancy,win_rate,max_dd,sharpe_ratio,profit_factor,num_trades,return_pct,run_timestamp
0,8040,13,28,14,3.0,2.0,1,0.00015,XAUUSD,M15,-0.020859,29.616464,0.291047,-1.423006,0.895345,5346,-0.222941,2026-08-17 08:03:12
1,8070,13,28,18,3.0,2.0,1,0.00015,XAUUSD,M15,-0.021527,29.573795,0.293988,-1.475108,0.891911,5374,-0.231291,2026-08-17 08:03:24
2,8039,13,28,14,3.0,2.0,0,0.00015,XAUUSD,M15,-0.021196,29.154105,0.310767,-1.521014,0.890099,5640,-0.239013,2026-08-17 08:03:12
3,8100,13,28,21,3.0,2.0,1,0.00015,XAUUSD,M15,-0.022358,29.462963,0.305099,-1.548781,0.887947,5401,-0.241429,2026-08-17 08:03:35
4,8038,13,28,14,3.0,1.5,1,0.00015,XAUUSD,M15,-0.020812,32.123999,0.305138,-1.565986,0.895672,5872,-0.244338,2026-08-17 08:03:11
5,7950,13,26,14,3.0,2.0,1,0.00015,XAUUSD,M15,-0.022659,29.442826,0.309190,-1.587627,0.884896,5493,-0.248845,2026-08-17 08:02:37
6,8069,13,28,18,3.0,2.0,0,0.00015,XAUUSD,M15,-0.021973,29.126384,0.315528,-1.588227,0.886056,5690,-0.249966,2026-08-17 08:03:23
7,8068,13,28,18,3.0,1.5,1,0.00015,XAUUSD,M15,-0.021064,32.327150,0.315983,-1.602064,0.894401,5931,-0.249787,2026-08-17 08:03:23
8,7980,13,26,18,3.0,2.0,1,0.00015,XAUUSD,M15,-0.023095,29.332850,0.308504,-1.624207,0.882824,5517,-0.254748,2026-08-17 08:02:49
9,8098,13,28,21,3.0,1.5,1,0.00015,XAUUSD,M15,-0.021352,32.267247,0.321270,-1.637875,0.893113,5973,-0.254989,2026-08-17 08:03:34


In [12]:
vwap_comparison = pd.read_sql_query(
    f"""SELECT use_vwap,
              COUNT(*) AS n,
              AVG(expectancy) AS mean_expectancy,
              AVG(win_rate) AS mean_win_rate,
              AVG(max_dd) AS mean_max_dd,
              AVG(sharpe_ratio) AS mean_sharpe,
              AVG(profit_factor) AS mean_profit_factor,
              AVG(num_trades) AS mean_num_trades,
              AVG(return_pct) AS mean_return_pct
       FROM {TABLE_NAME}
       WHERE sharpe_ratio IS NOT NULL
       GROUP BY use_vwap""",
    conn,
)
print(vwap_comparison)


   use_vwap     n  mean_expectancy  mean_win_rate  mean_max_dd  mean_sharpe  \
0         0  4050        -0.030004      36.786607     7.166232   -46.636949   
1         1  4050        -0.029124      40.417230     4.575989   -33.425846   

   mean_profit_factor  mean_num_trades  mean_return_pct  
0            0.567729    116973.174321        -7.160468  
1            0.608089     75573.305432        -4.567836  


In [ ]:
n1n2_comparison = pd.read_sql_query(
    """
    SELECT n1, n2,
           COUNT(*) AS n,
           AVG(expectancy) AS mean_expectancy,
           AVG(win_rate) AS mean_win_rate,
           AVG(max_dd) AS mean_max_dd,
           AVG(sharpe_ratio) AS mean_sharpe,
           AVG(profit_factor) AS mean_profit_factor,
           AVG(num_trades) AS mean_num_trades,
           AVG(return_pct) AS mean_return_pct
    FROM """ + TABLE_NAME + """
    WHERE sharpe_ratio IS NOT NULL
    GROUP BY n1, n2
    ORDER BY n1, n2
    """,
    conn,
)
print(n1n2_comparison)

    n1  n2 timeframe    n  mean_expectancy  mean_win_rate  mean_max_dd  \
0    5  18        M1  270        -0.030219      36.491276     6.447937   
1    5  20        M1  270        -0.030285      36.685776     6.351545   
2    5  22        M1  270        -0.030173      36.932960     6.265024   
3    5  24        M1  270        -0.029721      37.194295     6.184088   
4    5  26        M1  270        -0.029891      37.335139     6.133100   
5    5  28        M1  270        -0.029979      37.525030     6.073179   
6    7  18        M1  270        -0.030023      37.520288     6.151343   
7    7  20        M1  270        -0.029801      37.823765     6.065815   
8    7  22        M1  270        -0.029836      38.027758     5.994662   
9    7  24        M1  270        -0.029787      38.198511     5.943182   
10   7  26        M1  270        -0.029851      38.376852     5.894484   
11   7  28        M1  270        -0.029736      38.552006     5.841119   
12   9  18        M1  270        -0.02

In [ ]:
db_path = Path("../backtest_results.db")
conn = sqlite3.connect(db_path)


n1n2_comparison_m5 = pd.read_sql_query(
    """
    SELECT n1, n2,
           AVG(expectancy) AS mean_expectancy,
           AVG(win_rate) AS mean_win_rate,
           AVG(max_dd) AS mean_max_dd,
           AVG(sharpe_ratio) AS mean_sharpe,
           AVG(profit_factor) AS mean_profit_factor,
           AVG(num_trades) AS mean_num_trades,
           AVG(return_pct) AS mean_return_pct
    FROM """ + TABLE_NAME + """
    WHERE sharpe_ratio IS NOT NULL AND timeframe like 'M5'
    GROUP BY n1, n2
    ORDER BY mean_win_rate, mean_expectancy
    """,
    conn,
)
print(n1n2_comparison_m5)

    n1  n2   n  mean_expectancy  mean_win_rate  mean_max_dd  mean_sharpe  \
0    5  18  90        -0.030230      39.089599     3.062700   -21.317238   
1    5  20  90        -0.030008      39.416070     2.998167   -20.888108   
2    5  22  90        -0.029847      39.750780     2.938770   -20.495777   
3    5  24  90        -0.029758      39.973708     2.892872   -20.176707   
4    5  26  90        -0.029901      40.134849     2.878438   -19.997920   
5    7  18  90        -0.029960      40.218202     2.893254   -20.219858   
6    5  28  90        -0.029807      40.379205     2.846270   -19.779958   
7    7  20  90        -0.029781      40.605562     2.833578   -19.761846   
8    7  22  90        -0.029489      40.936904     2.775912   -19.354355   
9    9  18  90        -0.029753      41.121740     2.775606   -19.372364   
10   7  24  90        -0.029582      41.151349     2.758111   -19.230210   
11   7  26  90        -0.029697      41.303888     2.744449   -19.143704   
12   7  28  